# 04 — Paper and artifact discrepancies

**Workflow version:** 0.5.0

Compare the strict COBRApy reproduction with the paper's reported benchmark values and maintain a transparent discrepancy register. This notebook should be run only after notebooks 02 and 03 have produced their local result files.

In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RESULTS_DIR = ROOT / "results"

required = [
    RESULTS_DIR / "02_benchmark_summary.csv",
    RESULTS_DIR / "02_pairwise_comparison.csv",
    RESULTS_DIR / "03_structural_curation_audit.csv",
]
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 02 and 03 first: " + ", ".join(path.name for path in missing))

reproduced = pd.read_csv(RESULTS_DIR / "02_benchmark_summary.csv")
pairwise = pd.read_csv(RESULTS_DIR / "02_pairwise_comparison.csv")
structural = pd.read_csv(RESULTS_DIR / "03_structural_curation_audit.csv")

In [ ]:
paper = pd.DataFrame([
    {"model": "Yeast9", "paper_correct": 93, "paper_n": 147, "paper_accuracy": 93 / 147, "paper_ko_dead": 107},
    {"model": "Yeast9_curated", "paper_correct": 117, "paper_n": 147, "paper_accuracy": 117 / 147, "paper_ko_dead": 119},
])

comparison = reproduced.merge(paper, on="model", validate="one_to_one")
comparison["correct_delta"] = comparison["correct"] - comparison["paper_correct"]
comparison["accuracy_delta"] = comparison["accuracy"] - comparison["paper_accuracy"]

display(comparison)
comparison.to_csv(RESULTS_DIR / "04_paper_vs_reproduced_summary.csv", index=False)

In [ ]:
changes = pairwise.loc[
    pairwise["change"].isin(["fixed", "regression"]),
    [
        "pair_id", "excel_row_original", "gene_field_original", "chemical_original",
        "classification_original", "classification_curated",
        "ko_growth_original", "ko_growth_curated",
        "rescue_growth_original", "rescue_growth_curated", "change",
    ],
].copy()

display(changes)
changes.to_csv(RESULTS_DIR / "04_fixed_and_regressed_pairs.csv", index=False)

In [ ]:
discrepancy_register = pd.DataFrame([
    {
        "topic": "MET13 / r_0080",
        "paper_or_artifact_observation": "Table 1 and prose should be checked against each other and against the released original GPR.",
        "status": "manual source comparison required",
    },
    {
        "topic": "ALD2/ALD3 / r_0172",
        "paper_or_artifact_observation": "Compare the curated GPR in the released SBML with the curation described in the article and released MATLAB code.",
        "status": "structural audit available",
    },
    {
        "topic": "CYS4 / r_4702-r_4703",
        "paper_or_artifact_observation": "Compare which reaction is constrained in the paper description versus the released curated SBML.",
        "status": "structural audit available",
    },
    {
        "topic": "Solver implementation",
        "paper_or_artifact_observation": "Paper uses MATLAB/COBRA Toolbox/Gurobi; current reproduction uses COBRApy/GLPK.",
        "status": "solver sensitivity pending",
    },
    {
        "topic": "Shared-model state",
        "paper_or_artifact_observation": "Exploratory shared-state batches produced a THI6 inconsistency; strict pair isolation is now the reference workflow.",
        "status": "pipeline issue resolved by design",
    },
])

display(discrepancy_register)
discrepancy_register.to_csv(RESULTS_DIR / "04_discrepancy_register.csv", index=False)

## Interpretation rule

A mismatch becomes a scientific reproducibility discrepancy only after the local implementation, model identity, dataset parsing, solver status, pair isolation, and solver sensitivity have been checked. Keep **paper text**, **released SBML**, **released MATLAB code**, and **our COBRApy reproduction** as separate evidence layers.